In [1]:
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

sys.path.insert(0, '/Users/pbuathong/Desktop/Paul_work/bott')

/opt/anaconda3/envs/bott/bin/python
3.10.17 | packaged by conda-forge | (main, Apr 10 2025, 22:23:34) [Clang 18.1.8 ]
sys.version_info(major=3, minor=10, micro=17, releaselevel='final', serial=0)


In [2]:
import torch
from botorch.exceptions import InputDataWarning

from bott.optimization import run_one_trial, parse
from bott.physics_models import simulate_cbed
from bott.problem import OptimizationProblem
from bott.io import load_img, load_tif
from bott.utils import print_system_info, normalize
from botorch.acquisition.objective import GenericMCObjective, MCAcquisitionObjective
import glob, os, re
from botorch.models.transforms import Standardize
from botorch.models.transforms.input import Normalize
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.fit import fit_gpytorch_mll
from botorch.models.gp_regression import SingleTaskGP
from botorch.sampling.normal import SobolQMCNormalSampler
from torch.nn.functional import mse_loss

from scipy import ndimage
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
device = 'cpu' #torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
print(device)

cpu


In [3]:
img_dir = '/Users/pbuathong/Desktop/Paul_work/bott/investigation/images'
img_list = glob.glob(os.path.join(img_dir, '*.tif'))
def extract_params_from_filename(path):
    # Use regex to find numeric values after each keyword
    match = re.search(r'Thickness_(\d+)_TiltX_(-?\d+)_TiltY_(-?\d+)', path)
    if match:
        thickness = int(match.group(1))
        tiltx = int(match.group(2))
        tilty = int(match.group(3))
        return torch.tensor([thickness, tiltx, tilty], dtype=torch.float)
    else:
        raise ValueError(f"Could not parse path: {path}")
input_x = torch.stack([extract_params_from_filename(p) for p in img_list])  
plt.close('all')
%matplotlib inline
#widget

In [4]:
# Define the ground truth image and the input parameters
ground_truth = torch.Tensor(load_tif(img_list[2983]))# shape [157,157]
input_true = input_x[2983].clone()
# OptimizationProblem would keep all the tensor on the specified device
problem = OptimizationProblem(ground_truth=ground_truth,
                              output_path='./output/', 
                              save_results=True, 
                              reduction_params={'reduction_type':'square','reduce':'sum', 
                                                'reduction_kwargs':{'num_tiles':4}},
                              loss_params={'loss_type':'SSE', 'dp_pow': 1}, 
                              norm_arr=True, # normalize the pixels values
                              dim=3,#3, 
                              bounds=[(10,500), (-20, 20), (-20,20)],
                              noise_std=0,
                              dtype=torch.float64, 
                              device='cpu'
                              ) # "cpu" or "cuda" for physics simulation

loss_func = problem.loss_func
reduction_true = problem.reduction_true.to(torch.float64) #patch term
measurement_true = problem.measurement_true.to(torch.float64) #ground_truth
sf_tile = problem.scaling_factor

In [5]:
#patch size 3
imgs_total = []
arr_patch_loss = np.array([])
arr_y_value_CF = []
arr_pix_loss = np.array([])
for path in tqdm(img_list):
    temp_img = torch.Tensor(load_tif( path ))
    imgs_total.append(temp_img)
    #pixel SSE
    temp_pixel_loss = (temp_img-measurement_true).pow(2).sum(dim=(0,1))
    #loss_func(y_simu=temp_img,y_true=measurement_true,reduce=False)
    ## works fine as well for SSE for torch.Size([150,150])
    arr_pix_loss = np.append(arr_pix_loss,temp_pixel_loss)
    
    #patch term (w/o correction term)
    temp_patch = problem.reduction_func(temp_img)

    #patch SSE
    temp_patch_loss = loss_func(y_simu=temp_patch,
                                        y_true=reduction_true,
                                        reduce=False) #working correctly for SSE
    arr_patch_loss = np.append(arr_patch_loss, temp_patch_loss)
    #epsilon
    epsilon = temp_pixel_loss - sf_tile*temp_patch_loss

    #patch term (w/ correction term)
    temp_patches = torch.cat((temp_patch,epsilon.unsqueeze(0)),dim=-1)
    arr_y_value_CF.append(temp_patches.unsqueeze(0))

obj_real =  (sf_tile/sf_tile)*loss_func(y_simu=reduction_true,
                            y_true=reduction_true,
                            reduce=False)
arr_y_value_CF = torch.cat(arr_y_value_CF, dim=0) #99,10
pixel_SSE = torch.Tensor(arr_pix_loss).unsqueeze(-1)

100%|██████████| 4425/4425 [00:02<00:00, 1518.10it/s]


In [8]:
from botorch.acquisition import PosteriorMean as GPPosteriorMean
from botorch.optim import optimize_acqf
from botorch.acquisition.objective import GenericMCObjective
from bott.posterior_mean_gpcf import PosteriorMean as GPCFPosteriorMean


In [ ]:
num_X = input_x.shape[0]
num_train = 100
for seed in range(1):
    torch.manual_seed(seed*5)  # For reproducibility

    # Random permutation of indices #TODO: this doesn't sample well. stratified or clustering?
    perm = torch.randperm(num_X)
    train_indices = perm[:num_train]
    # Split data
    train_X_GP = input_x[train_indices]
    train_Y_GP = -1*pixel_SSE[train_indices]#.to(torch.float32) #pixel SSE
    #BO
    #train the model
    model = SingleTaskGP(train_X=train_X_GP.to(torch.float64), 
                        train_Y=train_Y_GP.to(torch.float64), 
                        train_Yvar=torch.ones_like(train_Y_GP).to(torch.float64) * 1e-6, #TODO Need to configure this variance scaling hyperparameter
                        outcome_transform=Standardize(m=train_Y_GP.shape[-1]),
                        input_transform=Normalize(d=train_X_GP.shape[-1])).to(device)
    fit_gpytorch_mll(ExactMarginalLogLikelihood(model.likelihood, model))

    # Define the acquisition function (posterior mean of the GP) and optimize to find the point the model predicts to be the best (max negative SSE)
    posterior_mean_function_GP = GPPosteriorMean(model)
    new_x_GP, predicted_SSE_value_GP = optimize_acqf(acq_function=posterior_mean_function_GP,
                             bounds=problem.bounds,
                             q=1,
                             num_restarts=30,
                             raw_samples=300,
                             )
    # TODO: add code to run the physics model and get the true SSE value at the new_x_GP input

    # BOCF
    objective = GenericMCObjective(lambda Y, X=None: -1*(((sf_tile)*(Y[...,:-1] - reduction_true.unsqueeze(0)).pow(2).sum(dim = -1 ) ) + Y[...,-1]))
    train_X_GPCF = input_x[train_indices]
    train_y_GPCF = arr_y_value_CF[train_indices,...]
    modelCF = SingleTaskGP(train_X=train_X_GPCF.to(torch.float64), 
                        train_Y=train_y_GPCF.to(torch.float64), train_Yvar=None,# torch.ones_like(train_y_value) * 1e-6, #TODO Need to configure this variance scaling hyperparameter
                            outcome_transform=Standardize(m=train_y_GPCF.shape[-1]),
                            input_transform=Normalize(d=train_X_GPCF.shape[-1])).to(device)
    fit_gpytorch_mll(ExactMarginalLogLikelihood(modelCF.likelihood, modelCF))

    # Define the acquisition function (posterior mean of the GPCF) and optimize to find the point the GPCFmodel predicts to be the best (max negative SSE)
    qmc_sampler = SobolQMCNormalSampler(torch.Size([256])).to(dtype=torch.float64)
    posterior_mean_function_GPCF = GPCFPosteriorMean(modelCF,sampler=qmc_sampler,objective=objective)
    new_x_GPCF, predicted_SSE_value_GPCF = optimize_acqf(acq_function=posterior_mean_function_GPCF,
                             bounds=problem.bounds,
                             q=1,
                             num_restarts=30,
                             raw_samples=300,
                             )
    # TODO: add code to run the physics model and get the true SSE value at the new_x_GPCF input

/opt/anaconda3/envs/bott/lib/python3.10/site-packages/gpytorch/likelihoods/noise_models.py:150: NumericalWarning: Very small noise values detected. This will likely lead to numerical instabilities. Rounding small noise values up to 1e-06.
  warnings.warn(


In [23]:
new_x_GP,new_x_GPCF,input_true

(tensor([[68.3875,  6.1540,  6.8553]], dtype=torch.float64),
 tensor([[42.5253,  7.3423, 11.3562]], dtype=torch.float64),
 tensor([51., -3.,  5.]))

In [ ]:
# measure how close the new_x_GP and new_x_GPCF are to the true input
distance_from_true_x_GP = torch.norm(new_x_GP - input_true, p=2)
distance_from_true_x_GPCF = torch.norm(new_x_GPCF - input_true, p=2)
print(f'distance from true input --  x GP: {distance_from_true_x_GP:.4f} and distance from true input -- x GPCF: {distance_from_true_x_GPCF:.4f}')


In [ ]:
# TODO add code to compare the pixel SSE of the new_x_GP and new_x_GPCF to the ground truth pixel SSE

distance from true input --  x GP: 19.7374 and distance from true input -- x GPCF: 14.8049
